# Kuliah #08 - Transformasi Basis: Pembuktian Konsep dan Implementasi Matriks

Notebook ini disusun sebagai pendamping modul kuliah **Serial Mekanika Kuantum Minimalis 2.0: Kuliah #08 - Transformasi Basis**. 

Secara struktur, notebook ini dibagi menjadi beberapa bagian utama sesuai dengan sub-bab di dalam diktat:
1. **Cara "Primitif" Mengubah Basis**: Penjabaran konversi representasi keadaan kuantum dan operator dari basis $HV$ ke basis $\pm 45$ melalui ekspansi *inner product* dan sisipan operator identitas (Persamaan 1 – 24).
2. **Transformasi Keserupaan (*Similarity Transformation*)**: Penurunan formal matriks transformasi uniter $\mathbf{U}$, aturan transformasi vektor keadaan $\vec{C}' = \mathbf{U}^\dagger \vec{C}$ dan operator $\mathbf{A}' = \mathbf{U}^\dagger \mathbf{A} \mathbf{U}$, serta bukti penjagaan sifat-sifat fisis (*trace*, determinan, sifat Hermitian, dan nilai *eigen*) (Persamaan 25 – 44).
3. **Aplikasi: Diagonalisasi**: Teorema spektral pada operator normal ($\hat{A}^\dagger\hat{A} = \hat{A}\hat{A}^\dagger$) dan diagonalisasi matriks operator proyeksi $\hat{P}_{+45}$ menggunakan basis *eigen*-nya (Persamaan 45 – 52).
4. **Latihan (Soal-Jawab)**:
   - **Soal 1**: Representasi vektor $|H\rangle$ dan $|V\rangle$ dalam basis polarisasi melingkar ($|L\rangle, |R\rangle$) serta pembuktian ortogonalitasnya.
   - **Soal 2**: Representasi matriks operator proyeksi $\hat{P}_H$ dan $\hat{P}_V$ dalam basis $LR$ dan verifikasi relasi aljabar idempoten serta ortogonalitas operator.
   - **Soal 3**: Analisis nilai *eigen* dan vektor *eigen* dari $\hat{P}_{+45}$ dalam basis $\pm 45$ beserta interpretasi fisis pengukuran ideal.
   - **Soal 4**: Penurunan representasi matriks operator rotasi polarisasi $\hat{R}_p(\theta)$ dalam basis $\pm 45$.
   - **Soal 5**: Pembuktian aljabar bahwa $\hat{R}_p(45^\circ)|+45\rangle = |V\rangle$ menggunakan representasi matriks pascatransformasi basis $\pm 45$.

Semua pembuktian matematis dijabarkan secara rinci menggunakan aljabar matriks dan vektor kolom/baris yang diturunkan langsung dari aljabar Dirac. Kode Python menggunakan modul `numpy` dan `sympy` disediakan untuk memverifikasi setiap konsep secara numerik maupun simbolik.

In [ ]:
import math
import numpy as np
import sympy as sp

np.set_printoptions(precision=4, suppress=True)
sp.init_printing()

def clean_array(A, tol=1e-12):
    """Membersihkan nilai elemen matriks/vektor yang mendekati nol akibat kesalahan pembulatan numerik (floating-point)."""
    A = np.array(A, dtype=complex)
    A[np.abs(A.real) < tol] = 1j * A[np.abs(A.real) < tol].imag
    A[np.abs(A.imag) < tol] = A[np.abs(A.imag) < tol].real
    if np.all(np.abs(A.imag) < tol):
        A = A.real
    return A

def print_matrix(name, M):
    M = clean_array(M)
    print(f"{name} =")
    print(M)
    print()

def print_ket(name, v):
    v = clean_array(v)
    print(f"|{name}> =")
    print(v)
    print()

# Fungsi perkalian dalam <bra|ket> dan norma
def inner_product(bra, ket):
    bra = np.array(bra, dtype=complex).flatten()
    ket = np.array(ket, dtype=complex).flatten()
    return np.vdot(bra, ket)

def norm(v):
    return np.sqrt(np.abs(inner_product(v, v)))

# Operasi Adjoint dan Outer Product
def adjoint(M):
    return np.conjugate(np.transpose(M))

def outer_product(ket, bra):
    ket = np.array(ket, dtype=complex).reshape(-1, 1)
    bra = np.array(bra, dtype=complex).reshape(1, -1)
    return ket @ np.conjugate(bra)

# Vektor Keadaan Dasar dalam Basis Horizontal-Vertikal (HV)
ket_H = np.array([[1], [0]], dtype=complex)
ket_V = np.array([[0], [1]], dtype=complex)

ket_plus45  = (1 / np.sqrt(2)) * np.array([[1], [1]], dtype=complex)
ket_minus45 = (1 / np.sqrt(2)) * np.array([[1], [-1]], dtype=complex)

ket_L = (1 / np.sqrt(2)) * np.array([[1], [1j]], dtype=complex)
ket_R = (1 / np.sqrt(2)) * np.array([[1], [-1j]], dtype=complex)

# Proyektor dalam Basis HV
P_H = outer_product(ket_H, ket_H)
P_V = outer_product(ket_V, ket_V)
P_plus45_HV  = outer_product(ket_plus45, ket_plus45)
P_minus45_HV = outer_product(ket_minus45, ket_minus45)

print("Setup selesai. Keadaan basis dan fungsi dasar siap digunakan.")

# 1. Cara "Primitif" Mengubah Basis

## 1.1 Penulisan Ulang Persamaan

Misalkan sebuah keadaan $|\psi\rangle$ diketahui dalam basis $HV$:
$$|\psi\rangle = c_H |H\rangle + c_V |V\rangle. \tag{1}$$

Kita ingin mentransformasikan $|\psi\rangle$ ke basis $\pm 45$, yaitu mencari koefisien $c_{+45}$ dan $c_{-45}$ dalam uraian:
$$|\psi\rangle = c_{+45} |+45\rangle + c_{-45} |-45\rangle. \tag{2}$$

Diketahui hubungan antara vektor-vektor basis tersebut:
$$|+45\rangle = \frac{1}{\sqrt{2}}(|H\rangle + |V\rangle), \qquad |-45\rangle = \frac{1}{\sqrt{2}}(|H\rangle - |V\rangle). \tag{3}$$

Berdasarkan definisi produk dalam (*inner product*), koefisien ekspansi diberikan oleh:
$$c_{+45} = \langle +45|\psi\rangle, \tag{4}$$
$$c_{-45} = \langle -45|\psi\rangle. \tag{5}$$

Substitusi keadaan $|\psi\rangle$ dari Persamaan (1) memberikan:
$$c_{+45} = \langle +45| (c_H|H\rangle + c_V|V\rangle) = c_H \langle +45|H\rangle + c_V \langle +45|V\rangle, \tag{6}$$
$$c_{-45} = \langle -45| (c_H|H\rangle + c_V|V\rangle) = c_H \langle -45|H\rangle + c_V \langle -45|V\rangle. \tag{7}$$

Mengambil konjugat kompleks dari Persamaan (3), kita mendapati *inner product* antar-basis:
$$\langle +45|H\rangle = \frac{1}{\sqrt{2}}, \quad \langle +45|V\rangle = \frac{1}{\sqrt{2}}, \quad \langle -45|H\rangle = \frac{1}{\sqrt{2}}, \quad \langle -45|V\rangle = -\frac{1}{\sqrt{2}}. \tag{8}$$

Substitusi nilai-nilai ini menghasilkan koefisien target:
$$c_{+45} = \frac{1}{\sqrt{2}}(c_H + c_V), \tag{9}$$
$$c_{-45} = \frac{1}{\sqrt{2}}(c_H - c_V). \tag{10}$$

### Metode Alternatif: Menyisipkan Operator Identitas $\hat{1}$
Kita juga dapat memulai dari definisi $c_{+45} = \langle +45|\psi\rangle$ dan menyisipkan operator identitas $\hat{1} = |H\rangle\langle H| + |V\rangle\langle V|$:
$$c_{+45} = \langle +45| \hat{1} |\psi\rangle = \langle +45| (|H\rangle\langle H| + |V\rangle\langle V|) |\psi\rangle = \langle +45|H\rangle\langle H|\psi\rangle + \langle +45|V\rangle\langle V|\psi\rangle = \frac{1}{\sqrt{2}}(c_H + c_V). \tag{11-12}$$

---

## 1.2 Contoh Transformasi Representasi Keadaan dan Operator

### Contoh 1: Representasi $|L\rangle$ dalam basis $\pm 45$
Kita mencari uraian $|L\rangle = c_{+45}|+45\rangle + c_{-45}|-45\rangle$ (Persamaan 13 & 14).
Diketahui hubungan basis: $|L\rangle = \frac{1}{\sqrt{2}}(|H\rangle + i|V\rangle)$, sehingga $\langle H|L\rangle = \frac{1}{\sqrt{2}}$ dan $\langle V|L\rangle = \frac{i}{\sqrt{2}}$ (Persamaan 15 & 16).
Dengan menyisipkan operator identitas:
$$c_{+45} = \langle +45|\hat{1}|L\rangle = \langle +45|H\rangle\langle H|L\rangle + \langle +45|V\rangle\langle V|L\rangle = \left(\frac{1}{\sqrt{2}}\right)\left(\frac{1}{\sqrt{2}}\right) + \left(\frac{1}{\sqrt{2}}\right)\left(\frac{i}{\sqrt{2}}\right) = \frac{1}{2}(1 + i). \tag{17}$$
Dengan cara serupa, $c_{-45} = \frac{1}{2}(1 - i)$. Persamaan uraian $|L\rangle$ menjadi:
$$|L\rangle = \frac{1}{2}(1+i)|+45\rangle + \frac{1}{2}(1-i)|-45\rangle = \frac{e^{i\pi/4}}{\sqrt{2}} (|+45\rangle - i|-45\rangle). \tag{18}$$

### Contoh 2: Transformasi Basis Operator Proyeksi $\hat{P}_H$ dari $HV$ ke $\pm 45$
Dalam basis $HV$, $\hat{P}_H \doteq \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}_{HV}$ (Persamaan 19).
Dalam basis $\pm 45$ (menggunakan urutan $|1\rangle = |+45\rangle, |2\rangle = |-45\rangle$):
$$\hat{P}_H \doteq \begin{pmatrix} \langle +45|\hat{P}_H|+45\rangle & \langle +45|\hat{P}_H|-45\rangle \\ \langle -45|\hat{P}_H|+45\rangle & \langle -45|\hat{P}_H|-45\rangle \end{pmatrix}_{45}. \tag{20}$$
Dengan mengekspansi $|\pm 45\rangle$ ke basis $HV$:
$$\langle +45|\hat{P}_H|+45\rangle = \frac{1}{2} (\langle H| + \langle V|) \hat{P}_H (|H\rangle + |V\rangle) = \frac{1}{2}(\langle H|\hat{P}_H|H\rangle + 0 + 0 + 0) = \frac{1}{2}(1) = \frac{1}{2}. \tag{21-22}$$
Dengan menghitung seluruh elemen, diperoleh representasi matriks baru:
$$\hat{P}_H \doteq \frac{1}{2} \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}_{45}. \tag{23}$$

Efek operator ini pada $|+45\rangle$ dalam basis baru adalah:
$$\hat{P}_H|+45\rangle \doteq \frac{1}{2}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}_{45} \begin{pmatrix} 1 \\ 0 \end{pmatrix}_{45} = \frac{1}{2}\begin{pmatrix} 1 \\ 1 \end{pmatrix}_{45} = \frac{1}{2}(|+45\rangle + |-45\rangle) = \frac{1}{\sqrt{2}}|H\rangle. \tag{24}$$

In [ ]:
# Demonstrasi Numerik & Simbolik Bagian 1: Cara Primitif Mengubah Basis

print("--- 1. Verifikasi Koefisien c_+45 dan c_-45 untuk sembarang |psi> ---")
psi_test = 0.6 * ket_H + 0.8j * ket_V # Keadaan sembarang dalam basis HV
c_plus_primitif  = inner_product(ket_plus45, psi_test)
c_minus_primitif = inner_product(ket_minus45, psi_test)

print(f"c_+45 (dari <+45|psi>) = {c_plus_primitif:.4f}")
print(f"c_-45 (dari <-45|psi>) = {c_minus_primitif:.4f}")
# Verifikasi rekonstruksi |psi> dalam basis +-45
psi_rekonstruksi = c_plus_primitif * ket_plus45 + c_minus_primitif * ket_minus45
print("Apakah rekonstruksi |psi> cocok dengan asal?", np.allclose(psi_test, psi_rekonstruksi))
print()

print("--- 2. Verifikasi Transformasi |L> ke Basis +-45 (Persamaan 17-18) ---")
c_plus_L  = inner_product(ket_plus45, ket_L)
c_minus_L = inner_product(ket_minus45, ket_L)
print(f"c_+45 untuk |L> = {c_plus_L:.4f} (Analitik: 0.5 + 0.5j = {0.5+0.5j})")
print(f"c_-45 untuk |L> = {c_minus_L:.4f} (Analitik: 0.5 - 0.5j = {0.5-0.5j})")
print()

print("--- 3. Konstruksi Primitif Matriks P_H dalam Basis +-45 (Persamaan 20-23) ---")
basis_45 = [ket_plus45, ket_minus45]
P_H_in_45 = np.zeros((2, 2), dtype=complex)
for i in range(2):
    for j in range(2):
        P_H_in_45[i, j] = inner_product(basis_45[i], P_H @ basis_45[j])

print_matrix("Matriks P_H dalam basis +-45", P_H_in_45)
print("Apakah sama dengan 0.5 * [[1, 1], [1, 1]]?", np.allclose(P_H_in_45, 0.5 * np.ones((2, 2))))

# 2. Transformasi Keserupaan (*Similarity Transformation*)

## 2.1 Penulisan Ulang Persamaan

Cara manual (primitif) di atas dapat diformulasikan secara elegan menggunakan sebuah **matriks transformasi $\mathbf{U}$**. Misalkan basis lama adalah $\{|u_i\rangle\}$ dan basis baru adalah $\{|v_i\rangle\}$. Dengan menyisipkan operator identitas $\hat{1} = \sum_j |u_j\rangle\langle u_j|$:
$$|v_i\rangle = \hat{1}|v_i\rangle = \sum_j |u_j\rangle\langle u_j|v_i\rangle. \tag{25}$$

Elemen baris ke-$j$ dan kolom ke-$i$ dari matriks transformasi $\mathbf{U}$ didefinisikan sebagai:
$$U_{ji} \equiv \langle u_j|v_i\rangle. \tag{26}$$

Artinya, **kolom ke-$i$ dari matriks $\mathbf{U}$ adalah koordinat vektor basis baru $|v_i\rangle$ ketika dinyatakan dalam basis lama $|u_j\rangle$**. Karena kedua himpunan basis ortonormal, matriks $\mathbf{U}$ bersifat **uniter** ($\mathbf{U}^\dagger \mathbf{U} = \mathbf{I}$), dengan elemen *adjoint*:
$$(\mathbf{U}^\dagger)_{ij} = U_{ji}^* = \langle v_i|u_j\rangle. \tag{27}$$

### Transformasi Vektor Keadaan ($\vec{C}' = \mathbf{U}^\dagger \vec{C}$)
Jika $|\psi\rangle = \sum_i c_i |u_i\rangle = \sum_i c'_i |v_i\rangle$, hubungan koefisien pada basis baru diperoleh dari:
$$c'_i = \langle v_i|\psi\rangle = \sum_j \langle v_i|u_j\rangle\langle u_j|\psi\rangle = \sum_j (\mathbf{U}^\dagger)_{ij} c_j. \tag{28-29}$$

Dalam bentuk aljabar matriks vektor kolom:
$$\vec{C}' = \mathbf{U}^\dagger \vec{C}. \tag{30}$$

### Transformasi Keserupaan untuk Operator ($\mathbf{A}' = \mathbf{U}^\dagger \mathbf{A} \mathbf{U}$)
Elemen matriks operator $\hat{A}$ dalam basis baru adalah $A'_{ij} = \langle v_i|\hat{A}|v_j\rangle$. Menyisipkan dua operator identitas basis lama:
$$A'_{ij} = \sum_k \sum_l \langle v_i|u_k\rangle \langle u_k|\hat{A}|u_l\rangle \langle u_l|v_j\rangle = \sum_k \sum_l (\mathbf{U}^\dagger)_{ik} A_{kl} U_{lj}. \tag{35-36}$$

Persamaan ini merupakan perkalian tiga matriks:
$$\mathbf{A}' = \mathbf{U}^\dagger \mathbf{A} \mathbf{U}. \tag{37}$$

---

## 2.2 Penjagaan Sifat-Sifat Fisis (*Invariance*)
Transformasi keserupaan melestarikan realitas fisis sistem kuantum:
1. **Trace**: $\text{Tr}(\mathbf{A}') = \text{Tr}(\mathbf{U}^\dagger \mathbf{A} \mathbf{U}) = \text{Tr}(\mathbf{U} \mathbf{U}^\dagger \mathbf{A}) = \text{Tr}(\mathbf{I}\mathbf{A}) = \text{Tr}(\mathbf{A})$. (Persamaan 41)
2. **Determinan**: $\det(\mathbf{A}') = \det(\mathbf{U}^\dagger \mathbf{A} \mathbf{U}) = \det(\mathbf{U}^\dagger)\det(\mathbf{A})\det(\mathbf{U}) = \det(\mathbf{I})\det(\mathbf{A}) = \det(\mathbf{A})$. (Persamaan 42)
3. **Sifat Hermitian**: Jika $\mathbf{A}^\dagger = \mathbf{A}$, maka $(\mathbf{A}')^\dagger = (\mathbf{U}^\dagger \mathbf{A} \mathbf{U})^\dagger = \mathbf{U}^\dagger \mathbf{A}^\dagger \mathbf{U} = \mathbf{U}^\dagger \mathbf{A} \mathbf{U} = \mathbf{A}'$. (Persamaan 43)
4. **Nilai Eigen**: Persamaan karakteristik $\det(\mathbf{A}' - \lambda\mathbf{I}) = \det(\mathbf{U}^\dagger(\mathbf{A}-\lambda\mathbf{I})\mathbf{U}) = \det(\mathbf{A}-\lambda\mathbf{I}) = 0$. Nilai *eigen* $\lambda$ tidak berubah. (Persamaan 44)

In [ ]:
# Demonstrasi Numerik Bagian 2: Transformasi Keserupaan & Invariansi Fisis

# 1. Membangun Matriks Transformasi U dari HV ke +-45 (Persamaan 31 & 32)
# Kolom 1: |+45> dalam HV, Kolom 2: |-45> dalam HV
U_HV_to_45 = np.hstack([ket_plus45, ket_minus45])
print_matrix("Matriks Transformasi U (HV -> +-45)", U_HV_to_45)

# 2. Contoh Transformasi |R> dari HV ke +-45 (Persamaan 33 & 34)
C_R_HV = ket_R
C_R_45 = adjoint(U_HV_to_45) @ C_R_HV
print_ket("Vektor Koordinat |R> dalam basis +-45 (C'_R)", C_R_45)
print("Apakah C'_R sama dengan 0.5 * [[1-i], [1+i]]?", np.allclose(C_R_45, 0.5 * np.array([[1-1j], [1+1j]])))
print()

# 3. Contoh Transformasi Proyeksi P_H dari HV ke +-45 (Persamaan 38-40)
P_H_45_sim = adjoint(U_HV_to_45) @ P_H @ U_HV_to_45
print_matrix("Matriks P'_H = U^dagger @ P_H @ U", P_H_45_sim)
print()

# 4. Pembuktian Penjagaan Sifat-Sifat Fisis (Persamaan 41-44)
print("--- Pembuktian Invariansi Fisis pada Matriks Hermitian Acak ---")
A_rnd = np.array([[3.0, 1-2j], [1+2j, -1.0]], dtype=complex) # Matriks Hermitian
A_prime = adjoint(U_HV_to_45) @ A_rnd @ U_HV_to_45

tr_A = np.trace(A_rnd)
tr_Ap = np.trace(A_prime)
det_A = np.linalg.det(A_rnd)
det_Ap = np.linalg.det(A_prime)
evals_A = np.sort(np.linalg.eigvalsh(A_rnd))
evals_Ap = np.sort(np.linalg.eigvalsh(A_prime))

print(f"1. Trace: Tr(A) = {tr_A:.4f} | Tr(A') = {tr_Ap:.4f} | Sama? {np.isclose(tr_A, tr_Ap)}")
print(f"2. Determinan: det(A) = {det_A:.4f} | det(A') = {det_Ap:.4f} | Sama? {np.isclose(det_A, det_Ap)}")
print(f"3. Sifat Hermitian: Apakah A' == (A')^dagger? {np.allclose(A_prime, adjoint(A_prime))}")
print(f"4. Nilai Eigen A : {evals_A}")
print(f"   Nilai Eigen A': {evals_Ap}")
print(f"   Apakah nilai eigen identik? {np.allclose(evals_A, evals_Ap)}")

# 3. Aplikasi: Diagonalisasi Operator Normal

## 3.1 Penulisan Ulang Persamaan

Suatu operator $\hat{A}$ disebut **operator normal** apabila berkomutasi dengan *adjoint*-nya:
$$\hat{A}^\dagger \hat{A} = \hat{A} \hat{A}^\dagger. \tag{45}$$

Kelas operator ini mencakup operator Hermitian dan operator uniter. **Teorema spektral** menyatakan bahwa setiap operator normal selalu dapat didiagonalkan oleh transformasi uniter. Jika basis baru disusun dari vektor-vektor *eigen* ternormalisasi milik operator tersebut, matriks transformasinya menghasilkan bentuk diagonal $\mathbf{D}$:
$$\mathbf{U}^\dagger \mathbf{A} \mathbf{U} = \mathbf{D}, \qquad \text{atau} \qquad \mathbf{A} = \mathbf{U} \mathbf{D} \mathbf{U}^\dagger. \tag{46-47}$$

### Contoh: Diagonalisasi Matriks Proyeksi $\hat{P}_{+45}$ (Persamaan 48 – 52)
Dalam basis $HV$:
$$\hat{P}_{+45} \doteq \mathbf{P}_{+45} = \frac{1}{2} \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}_{HV}. \tag{48}$$

Vektor-vektor *eigen* ternormalisasi dari $\hat{P}_{+45}$ adalah:
$$|+45\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} \quad (\lambda_1 = 1), \qquad |-45\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix} \quad (\lambda_2 = 0). \tag{49}$$

Matriks transformasi $\mathbf{U}$ dan *adjoint*-nya yang disusun dari vektor *eigen*:
$$\mathbf{U} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad \mathbf{U}^\dagger = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}. \tag{50-51}$$

Melakukan transformasi keserupaan:
$$\mathbf{P}_{diag} = \mathbf{U}^\dagger \mathbf{P}_{+45} \mathbf{U} = \frac{1}{4}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 2 & 0 \\ 2 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}_{45}. \tag{52}$$

Elemen pada diagonal utama (1 dan 0) persis merupakan nilai-nilai *eigen* operator tersebut.

In [ ]:
# Demonstrasi Numerik Bagian 3: Diagonalisasi Operator Normal

print("--- Diagonalisasi Matriks Proyektor P_+45 ---")
# 1. Cari nilai eigen dan vektor eigen secara numerik
evals, evecs = np.linalg.eigh(P_plus45_HV)
# Urutkan dari nilai eigen terbesar (1 lalu 0)
idx = np.argsort(evals)[::-1]
evals = evals[idx]
U_diag = evecs[:, idx]

print("Nilai Eigen:", evals)
print_matrix("Matriks Transformasi Eigen U_diag", U_diag)

# 2. Lakukan transformasi U^dagger @ A @ U
P_diag_num = adjoint(U_diag) @ P_plus45_HV @ U_diag
print_matrix("Matriks Hasil Diagonalisasi P_diag", P_diag_num)
print("Apakah P_diag sama persis dengan diag(1, 0)?", np.allclose(P_diag_num, np.diag([1, 0])))

# 4. Latihan (Soal-Jawab)

## Soal 1
**Tentukan vektor-vektor kolom yang merepresentasikan keadaan terpolarisasi linier $|H\rangle$ dan $|V\rangle$ dengan menggunakan keadaan terpolarisasi melingkar $|L\rangle$ dan $|R\rangle$ sebagai basisnya. Selanjutnya, buktikan bahwa vektor-vektor $|H\rangle$ dan $|V\rangle$ di dalam basis $|L\rangle$ dan $|R\rangle$ itu bersifat ortogonal.**

### Jawaban Analitik:
Diketahui definisi basis melingkar dalam basis $HV$: $|L\rangle = \frac{1}{\sqrt{2}}(|H\rangle + i|V\rangle)$ dan $|R\rangle = \frac{1}{\sqrt{2}}(|H\rangle - i|V\rangle)$.
Menjumlahkan keduanya menghasilkan $|H\rangle$, sedangkan mengurangkannya menghasilkan $|V\rangle$:
$$|H\rangle = \frac{1}{\sqrt{2}}(|L\rangle + |R\rangle) \doteq \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}_{LR},$$
$$|V\rangle = -\frac{i}{\sqrt{2}}|L\rangle + \frac{i}{\sqrt{2}}|R\rangle \doteq \frac{1}{\sqrt{2}}\begin{pmatrix} -i \\ i \end{pmatrix}_{LR}.$$

Uji ortogonalitas dengan *inner product* $\langle H|V\rangle_{LR}$:
$$\langle H|V\rangle_{LR} = \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \end{pmatrix} \right] \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} -i \\ i \end{pmatrix} \right] = \frac{1}{2}(-i + i) = 0. \quad \blacksquare$$

---

## Soal 2
**Tentukan representasi matriks dari operator proyeksi $\hat{P}_H = |H\rangle\langle H|$ dan $\hat{P}_V = |V\rangle\langle V|$ dalam basis $|L\rangle$ dan $|R\rangle$. Periksalah hubungan $\hat{P}_H^2 = \hat{P}_H$, $\hat{P}_V^2 = \hat{P}_V$, dan $\hat{P}_H\hat{P}_V = \hat{P}_V\hat{P}_H = 0$ terpenuhi.**

### Jawaban Analitik:
Menggunakan vektor kolom basis $LR$ dari Soal 1, kita hitung *outer product*:
$$\hat{P}_H \doteq \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} \right] \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \end{pmatrix} \right] = \frac{1}{2}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}_{LR},$$
$$\hat{P}_V \doteq \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} -i \\ i \end{pmatrix} \right] \left[ \frac{1}{\sqrt{2}}\begin{pmatrix} i & -i \end{pmatrix} \right] = \frac{1}{2}\begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix}_{LR}.$$

Pembuktian hubungan relasi aljabar:
1. $\hat{P}_H^2 = \frac{1}{4}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix} = \frac{1}{4}\begin{pmatrix} 2 & 2 \\ 2 & 2 \end{pmatrix} = \hat{P}_H$. $\blacksquare$
2. $\hat{P}_V^2 = \frac{1}{4}\begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix}\begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix} = \frac{1}{4}\begin{pmatrix} 2 & -2 \\ -2 & 2 \end{pmatrix} = \hat{P}_V$. $\blacksquare$
3. $\hat{P}_H\hat{P}_V = \frac{1}{4}\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}\begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix} = \frac{1}{4}\begin{pmatrix} 0 & 0 \\ 0 & 0 \end{pmatrix} = 0$. $\blacksquare$

---

## Soal 3
**Hitunglah nilai-nilai *eigen* dan tentukan vektor-vektor *eigen* dari operator proyeksi $\hat{P}_{+45}$ dalam basis $\pm 45$. Apa makna fisis dari hasil ini?**

### Jawaban Analitik:
Dalam basis $\pm 45$, vektor basis langsung dinyatakan sebagai $|+45\rangle \doteq \begin{pmatrix} 1 \\ 0 \end{pmatrix}_{45}$ dan $|-45\rangle \doteq \begin{pmatrix} 0 \\ 1 \end{pmatrix}_{45}$.
Representasi proyektor menjadi matriks diagonal:
$$\hat{P}_{+45} \doteq \begin{pmatrix} 1 \\ 0 \end{pmatrix}\begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}_{45}.$$
Nilai *eigen* terbaca langsung dari diagonal utama: $\lambda_1 = 1$ (vektor *eigen* $|+45\rangle$) dan $\lambda_2 = 0$ (vektor *eigen* $|-45\rangle$).
**Makna Fisis**: Operator proyeksi merepresentasikan proses pengukuran ideal pada polarisator $+45^\circ$. Nilai *eigen* $1$ berarti probabilitas $100\%$ foton diteruskan (jika sejajar sumbu polarisator), sedangkan nilai *eigen* $0$ berarti $100\%$ foton diblokir (jika tegak lurus pada $-45^\circ$).

---

## Soal 4 & Soal 5
**Nyatakanlah operator rotasi polarisasi $\hat{R}_p(\theta)$ dalam basis $\pm 45$. Selanjutnya buktikan bahwa $\hat{R}_p(45^\circ)|+45\rangle = |V\rangle$.**

### Jawaban Analitik:
Matriks rotasi dalam basis $HV$ adalah $\mathbf{R}_{HV}(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$.
Matriks transformasi dari $HV$ ke $\pm 45$ adalah $\mathbf{U} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$. Melakukan transformasi keserupaan $\mathbf{R}_{45}(\theta) = \mathbf{U}^\dagger \mathbf{R}_{HV}(\theta) \mathbf{U}$:
$$\mathbf{R}_{45}(\theta) = \frac{1}{2}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}_{45}.$$

Untuk $\theta = 45^\circ$, matriksnya adalah $\mathbf{R}_{45}(45^\circ) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ -1 & 1 \end{pmatrix}_{45}$.
Aksi pada keadaan $|+45\rangle \doteq \begin{pmatrix} 1 \\ 0 \end{pmatrix}_{45}$:
$$\mathbf{R}_{45}(45^\circ)|+45\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ -1 & 1 \end{pmatrix}\begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix}_{45} = \frac{1}{\sqrt{2}}(|+45\rangle - |-45\rangle) = |V\rangle. \quad \blacksquare$$

In [ ]:
# Verifikasi Komputasi Latihan Soal 1 - 5

print("--- Verifikasi Soal 1: Vektor |H> dan |V> dalam Basis LR ---")
ket_H_LR = (1 / np.sqrt(2)) * np.array([[1], [1]], dtype=complex)
ket_V_LR = (1 / np.sqrt(2)) * np.array([[-1j], [1j]], dtype=complex)
print("Ortogonalitas <H|V> dalam basis LR =", inner_product(ket_H_LR, ket_V_LR))
print()

print("--- Verifikasi Soal 2: Matriks P_H dan P_V dalam Basis LR ---")
P_H_LR = outer_product(ket_H_LR, ket_H_LR)
P_V_LR = outer_product(ket_V_LR, ket_V_LR)
print_matrix("P_H dalam basis LR", P_H_LR)
print_matrix("P_V dalam basis LR", P_V_LR)
print("Apakah P_H^2 == P_H?", np.allclose(P_H_LR @ P_H_LR, P_H_LR))
print("Apakah P_V^2 == P_V?", np.allclose(P_V_LR @ P_V_LR, P_V_LR))
print("Apakah P_H @ P_V == 0?", np.allclose(P_H_LR @ P_V_LR, np.zeros((2, 2))))
print()

print("--- Verifikasi Soal 4 & 5: Rotasi R_p(theta) dalam Basis +-45 ---")
theta_sym = sp.Symbol('theta', real=True)
R_HV_sym = sp.Matrix([[sp.cos(theta_sym), -sp.sin(theta_sym)], [sp.sin(theta_sym), sp.cos(theta_sym)]])
U_sym = (sp.S(1)/sp.sqrt(2)) * sp.Matrix([[1, 1], [1, -1]])
R_45_sym = sp.simplify(U_sym.T * R_HV_sym * U_sym)

print("Matriks Rotasi Simbolik R_p(theta) dalam basis +-45:")
sp.pprint(R_45_sym)
print()

# Verifikasi Soal 5: R(45 deg) |+45> == |V>
R_45_num = (1 / np.sqrt(2)) * np.array([[1, 1], [-1, 1]])
ket_plus_in_45 = np.array([[1], [0]]) # |+45> dalam basis +-45 adalah [1, 0]^T
ket_out_in_45  = R_45_num @ ket_plus_in_45

# Konversi vektor keluaran dari basis +-45 ke basis HV untuk memastikan sama dengan |V>
ket_out_HV = U_HV_to_45 @ ket_out_in_45
print_ket("Keluaran R_p(45)|+45> pascakonversi ke basis HV", ket_out_HV)
print("Apakah tepat sama dengan |V>?", np.allclose(ket_out_HV, ket_V))